# Decision Tree baseline — Letter Recognition

Notebook cơ bản, có thể upload trực tiếp lên Kaggle. Mô hình dùng `DecisionTreeClassifier` với cùng protocol của project: `random_state=42`, test 20% và stratify.

**Kaggle accelerator:** chọn **None (CPU)**. Decision Tree của scikit-learn không dùng GPU; bật T4/P100 không làm bước huấn luyện này nhanh hơn.

## 1. Import và cấu hình

In [ ]:
import json
import platform
import time
from datetime import UTC, datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

RANDOM_STATE = 42
TEST_SIZE = 0.20
EXPERIMENT_ID = "dt_letter_baseline"
TARGET = "letter"
FEATURES = [
    "x_box", "y_box", "width", "high", "onpix", "x_bar",
    "y_bar", "x2bar", "y2bar", "xybar", "x2ybr", "xy2br",
    "x_ege", "xegvy", "y_ege", "yegvx",
]

IS_KAGGLE = Path("/kaggle/working").exists()
OUTPUT_ROOT = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
FIGURES_DIR = OUTPUT_ROOT / "figures"
RESULTS_DIR = OUTPUT_ROOT / "results"
MODELS_DIR = OUTPUT_ROOT / "models"
for directory in (FIGURES_DIR, RESULTS_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
print({"python": platform.python_version(), "sklearn": sklearn.__version__, "kaggle": IS_KAGGLE})

## 2. Tìm và đọc dữ liệu

Notebook ưu tiên cặp `train.csv`/`test.csv` chuẩn của project. Nếu chỉ có file raw, notebook loại exact duplicates rồi tạo split đúng protocol. Trên Kaggle, hãy **Add Input** chứa các CSV của project.

In [ ]:
REQUIRED_COLUMNS = set(FEATURES + [TARGET])
SEARCH_ROOTS = [Path.cwd(), Path.cwd().parent]
if Path("/kaggle/input").exists():
    SEARCH_ROOTS.insert(0, Path("/kaggle/input"))

def csv_has_columns(path):
    try:
        return REQUIRED_COLUMNS.issubset(pd.read_csv(path, nrows=1).columns)
    except (OSError, UnicodeError, ValueError, pd.errors.ParserError):
        return False

def find_canonical_split():
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for train_path in root.rglob("train.csv"):
            test_path = train_path.with_name("test.csv")
            if test_path.exists() and csv_has_columns(train_path) and csv_has_columns(test_path):
                return train_path, test_path
    return None

def find_raw_letter_file():
    names = ("letter_recognition.csv", "letter-recognition.data")
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for name in names:
            matches = list(root.rglob(name))
            if matches:
                return matches[0]
    return None

split_paths = find_canonical_split()
if split_paths:
    train_path, test_path = split_paths
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    data_source = f"canonical split: {train_path.parent}"
else:
    raw_path = find_raw_letter_file()
    if raw_path is None:
        raise FileNotFoundError(
            "Không tìm thấy Letter Recognition. Hãy Add Input chứa train.csv/test.csv "
            "hoặc letter_recognition.csv trên Kaggle."
        )
    if raw_path.name == "letter-recognition.data":
        raw_df = pd.read_csv(raw_path, header=None, names=[TARGET] + FEATURES)
        raw_df = raw_df[FEATURES + [TARGET]]
    else:
        raw_df = pd.read_csv(raw_path)
    raw_df = raw_df[FEATURES + [TARGET]].drop_duplicates().reset_index(drop=True)
    train_df, test_df = train_test_split(
        raw_df, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=raw_df[TARGET]
    )
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    data_source = f"raw CSV + notebook split: {raw_path}"

assert set(train_df.columns) == REQUIRED_COLUMNS
assert set(test_df.columns) == REQUIRED_COLUMNS
assert not train_df.isna().any().any() and not test_df.isna().any().any()
assert set(train_df[TARGET].unique()) == set(test_df[TARGET].unique())

print("Nguồn:", data_source)
print("Train/Test:", train_df.shape, test_df.shape)
display(train_df.head())

## 3. Tổng quan nhanh

In [ ]:
class_counts = train_df[TARGET].value_counts().sort_index()
ax = class_counts.plot(kind="bar", figsize=(12, 4), color="#2563EB")
ax.set(title="Letter Recognition - training class distribution", xlabel="Letter", ylabel="Samples")
plt.tight_layout()
plt.show()

display(train_df[FEATURES].describe().T.round(3))

## 4. Huấn luyện Decision Tree cơ bản

Không scale dữ liệu vì cây quyết định chỉ so sánh ngưỡng. Đây là baseline chưa pruning/tuning để làm mốc cho các thí nghiệm sau.

In [ ]:
X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

model = DecisionTreeClassifier(criterion="gini", random_state=RANDOM_STATE)
started = time.perf_counter()
model.fit(X_train, y_train)
training_seconds = time.perf_counter() - started

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
metrics = {
    "train_accuracy": accuracy_score(y_train, y_train_pred),
    "test_accuracy": accuracy_score(y_test, y_test_pred),
    "error_rate": 1 - accuracy_score(y_test, y_test_pred),
    "precision_macro": precision_score(y_test, y_test_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_test_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_test_pred, average="macro", zero_division=0),
    "training_seconds": training_seconds,
    "tree_depth": model.get_depth(),
    "leaf_count": model.get_n_leaves(),
}
display(pd.Series(metrics, name="value").to_frame().round(4))
display(pd.DataFrame(classification_report(y_test, y_test_pred, output_dict=True, zero_division=0)).T.round(3))

## 5. Confusion matrix và feature importance

In [ ]:
fig, ax = plt.subplots(figsize=(13, 11))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_test_pred, labels=model.classes_, cmap="Blues", colorbar=False, ax=ax
)
ax.set_title("Letter Recognition - Decision Tree baseline")
fig.tight_layout()
confusion_path = FIGURES_DIR / f"{EXPERIMENT_ID}__confusion_matrix.png"
fig.savefig(confusion_path, dpi=200, bbox_inches="tight")
plt.show()

importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(9, 6))
importance.plot.barh(ax=ax, color="#0F766E")
ax.set(title="Decision Tree - feature importance", xlabel="Gini importance", ylabel="")
fig.tight_layout()
importance_path = FIGURES_DIR / f"{EXPERIMENT_ID}__feature_importance.png"
fig.savefig(importance_path, dpi=200, bbox_inches="tight")
plt.show()
display(importance.sort_values(ascending=False).head(10).rename("importance").to_frame())

## 6. Trực quan ba tầng đầu của cây

In [ ]:
fig, ax = plt.subplots(figsize=(24, 11))
plot_tree(
    model, feature_names=FEATURES, class_names=model.classes_, max_depth=3,
    filled=True, rounded=True, fontsize=7, ax=ax
)
ax.set_title("Letter Recognition - first three levels of the baseline tree")
fig.tight_layout()
tree_path = FIGURES_DIR / f"{EXPERIMENT_ID}__tree_top_levels.png"
fig.savefig(tree_path, dpi=200, bbox_inches="tight")
plt.show()

## 7. Lưu model và result contract

In [ ]:
model_path = MODELS_DIR / f"{EXPERIMENT_ID}.joblib"
joblib.dump(model, model_path)

result = {
    "schema_version": "1.0",
    "experiment_id": EXPERIMENT_ID,
    "dataset": "letter_recognition",
    "model": "DecisionTreeClassifier",
    "split": {"test_size": TEST_SIZE, "random_state": RANDOM_STATE, "stratify": True},
    "metrics": {key: float(value) if isinstance(value, (float, np.floating)) else int(value) for key, value in metrics.items()},
    "artifacts": {
        "figure_paths": [str(confusion_path), str(importance_path), str(tree_path)],
        "model_path": str(model_path),
    },
    "notes": "CPU baseline; criterion=gini; no scaling; no pruning or hyperparameter tuning.",
    "created_at_utc": datetime.now(UTC).isoformat(),
}
result_path = RESULTS_DIR / f"{EXPERIMENT_ID}.json"
result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print("Model:", model_path)
print("Result:", result_path)
print("Figures:", *result["artifacts"]["figure_paths"], sep="\n- ")

## Kết luận baseline

So sánh `train_accuracy` và `test_accuracy` để nhận biết overfitting. Các bước tiếp theo phù hợp là depth sweep, `min_samples_leaf`, cost-complexity pruning và cross-validation; không dùng test set để chọn hyperparameter.